# 03 — When particles interact (Solution)

**Time:** 55 minutes
**Question:** can a poor fit be repaired by changing particle shape, or is
inter-particle physics missing?

Start with the simplest plausible particle model. Use the residual pattern as a
clue before adding complexity.

In [ ]:
from pathlib import Path
import sys

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'pyproject.toml').exists())
sys.path.insert(0, str(ROOT / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import sans_fitter
from sans_fitter import SANSFitter

DATA = ROOT / 'data'
print(f'Using sans_fitter from {Path(sans_fitter.__file__).resolve()}')
%matplotlib inline


In [ ]:
filename = DATA / 'interacting_spheres.csv'
values = np.genfromtxt(filename, delimiter=',', names=True)
fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(values['Q'], values['I'], yerr=values['dI'], fmt='o', ms=3)
ax.set(xscale='log', yscale='log', xlabel=r'$q$ / Å$^{-1}$', ylabel=r'$I(q)$')
ax.grid(alpha=0.25)
plt.show()


## Exercise 1 — Deliberately incomplete model

From the previous notebooks, build a dilute-sphere fit without copying a
complete solution. Locate the structured residual region. Why can changing
radius or scale not independently repair it?

In [ ]:
dilute = SANSFitter()
dilute.load_data(str(filename))
dilute.set_model('sphere')
dilute.set_param('sld', value=4.2, vary=False)
dilute.set_param('sld_solvent', value=1.0, vary=False)
dilute.set_param('radius', value=40, min=15, max=90, vary=True)
dilute.set_param('scale', value=0.05, min=0.001, max=0.2, vary=True)
dilute.set_param('background', value=0.002, min=0, max=0.02, vary=True)
dilute_result = dilute.fit(engine='lmfit', method='least_squares')
dilute.plot_results(show_residuals=True, log_scale=True)


## Exercise 2 — Add excluded-volume interactions

`set_structure_factor('hardsphere')` combines the selected form factor with a
hard-sphere interaction model. Calling `get_params()` afterward reveals the
additional parameters:

- `volfraction`: fraction of the sample volume occupied by interacting spheres;
- `radius_effective`: interaction distance, which may include a solvation layer.

Fit both independently. The mode-control parameters should remain fixed.

In [ ]:
interacting = SANSFitter()
interacting.load_data(str(filename))
interacting.set_model('sphere')
interacting.set_param('sld', value=4.2, vary=False)
interacting.set_param('sld_solvent', value=1.0, vary=False)
interacting.set_param('radius', value=40, min=15, max=90, vary=True)
interacting.set_param('scale', value=0.05, min=0.001, max=0.2, vary=True)
interacting.set_param('background', value=0.002, min=0, max=0.02, vary=True)
interacting.set_structure_factor('hardsphere')
interacting.set_param('radius_effective', value=45, min=20, max=100, vary=True)
interacting.set_param('volfraction', value=0.2, min=0.01, max=0.55, vary=True)
interacting_result = interacting.fit(engine='lmfit', method='least_squares')
interacting.plot_results(show_residuals=True, log_scale=True)


In [ ]:
print('Form factor only:', dilute_result['chisq'])
print('With structure factor:', interacting_result['chisq'])
for name in ['radius', 'radius_effective', 'volfraction']:
    print(name, interacting_result['parameters'][name]['formatted'])


## Exercise 3 — Interpret and challenge

1. What real-space correlation creates the peak?
2. Why might `radius_effective` differ from the form-factor radius?
3. What assumptions in the hard-sphere model could fail?
4. Try `radius_effective_mode='link_radius'`. Is that constraint supported by
   these data?

In [ ]:
interpretation = (
    'A form factor alone cannot reproduce the suppressed low-q intensity and '
    'correlation peak. The hard-sphere structure factor captures excluded-volume '
    'correlations; the fitted volume fraction should be treated as model-dependent.'
)
print(interpretation)


## Exit ticket

Sketch a decision tree for an unfamiliar curve: data checks, simplest form
factor, residual diagnosis, alternative shapes, q-range sensitivity, and only
then interaction models. Name one conclusion that SANS alone cannot establish.